In [0]:
from pyspark.sql.types import *
schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("order_date", StringType(), True),
    StructField("customer", ArrayType(StructType([
        StructField("customer_id", StringType(), True),
        StructField("name", StringType(), True),
        StructField("email", StringType(), True)])), True),
    StructField("items", ArrayType(StructType([
        StructField("item_id", StringType(), True),
        StructField("product_name", StringType(), True),
        StructField("category", StringType(), True),
        StructField("quantity", IntegerType(), True),
        StructField("price", FloatType(), True)
    ])), True),
    StructField("payment",ArrayType(StructType([
        StructField("payment_mode", StringType(), True),
        StructField("transaction_id", StringType(), True),
        StructField("status", StringType(), True)
    ])), True),
    StructField("delivery_status", StringType(), True)])

df =spark.read.format("json").schema(schema).load("/Volumes/cab_data_catalog/bronze/orders/orders.json")


In [0]:

df.select(
    "order_id",
    "order_date",
    "delivery_status"
).display()


In [0]:
df.printSchema()

In [0]:
from pyspark.sql.functions import *
final_df = df \
    .withColumn("customer", explode("customer")) \
    .withColumn("payment", explode("payment")) \
    .withColumn("item", explode("items")) \
    .select(
        "order_id",
        "order_date",
        col("customer.customer_id").alias("customer_id"),
        col("customer.name").alias("customer_name"),
        col("item.product_name").alias("product"),
        col("item.category").alias("category"),
        col("item.quantity").alias("quantity"),
        col("item.price").alias("price"),
        (col("item.quantity") * col("item.price")).alias("item_total"),
        col("payment.payment_mode").alias("payment_mode"),
        col("payment.status").alias("payment_status"),
        "delivery_status"
    )

final_df.display()

In [0]:
from pyspark.sql.functions import *
df.withColumn("test", explode("items"))
df.display()